# Speckle2Self: ultrasound speckle reduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tue-bmd/zea/blob/main/docs/source/notebooks/models/speckle2self_despeckling_example.ipynb) &nbsp; [![View on GitHub](https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github)](https://github.com/tue-bmd/zea/blob/main/docs/source/notebooks/models/speckle2self_despeckling_example.ipynb)

This notebook demonstrates how to perform **self-supervised ultrasound speckle reduction** using [Speckle2Self](https://arxiv.org/abs/2507.06828) within the [zea](https://github.com/tue-bmd/zea) framework.

We apply the model to an **in-vivo carotid artery** scan from [zeahub/zea-carotid-2023](https://huggingface.co/datasets/zeahub/zea-carotid-2023), matching the domain on which the `speckle2self-invivo` weights were trained.

### Speckle2Self
[![Hugging Face model](https://img.shields.io/badge/Hugging%20Face-Model-yellow?logo=huggingface)](https://huggingface.co/zeahub/speckle2self-invivo) &nbsp; [![arXiv](https://img.shields.io/badge/arXiv-Paper-b31b1b.svg)](https://arxiv.org/abs/2507.06828) &nbsp; [![GitHub](https://img.shields.io/badge/GitHub-Code-black?logo=github)](https://github.com/noseefood/speckle2self)

- Self-supervised: **no clean/noise-free training data required**.
- Works on single-channel ultrasound images.
- Best results on **envelope data** (before log-compression) at >= 512 x 512 resolution.

### Citation
If you use this model, please cite the Speckle2Self paper: [arXiv:2507.06828](https://arxiv.org/abs/2507.06828).

### Workflow
1. Load in-vivo carotid RF data and beamform to a linear envelope image (`Beamform` -> `EnvelopeDetect`).
2. Load the Speckle2Self Keras model and run inference on the linear envelope data.
3. Compare original and despeckled B-mode images.


‼️ **Important:** This notebook is optimized for **GPU/TPU**. Code execution on a **CPU** may be very slow.

If you are running in Colab, please enable a hardware accelerator via:

**Runtime → Change runtime type → Hardware accelerator → GPU/TPU** 🚀.

In [1]:
%%capture
%pip install zea

In [2]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import matplotlib.pyplot as plt

from zea import init_device
from zea.data import load_file
from zea.models.speckle2self import Speckle2Self
from zea.ops import Beamform, EnvelopeDetect, Pipeline
from zea.visualize import set_mpl_style

init_device(verbose=False)
set_mpl_style()

zea: Using backend 'tensorflow'


In [3]:
# Example parameters
carotid_path = "hf://zeahub/zea-carotid-2023/2_cross_bifur_right_0000_small.hdf5"
frame_idx = [0]  # load one frame for a quick example
n_tx = 11  # transmits per frame (fewer -> faster; more -> better quality)
dynamic_range = (-40, 0)  # dB

## Load carotid data

We load a single frame of an in-vivo carotid scan from [zeahub/zea-carotid-2023](https://huggingface.co/datasets/zeahub/zea-carotid-2023) on Hugging Face.

The pipeline stops **after envelope detection** (no `Normalize` / `LogCompress`) because Speckle2Self works on **linear-scale envelope data**.


In [4]:
data, scan, probe = load_file(carotid_path, "raw_data", indices=frame_idx)

scan.set_transmits(n_tx)
scan.zlims = (0, 0.04)
scan.xlims = probe.xlims
scan.n_ch = data.shape[-1]  # RF data: channels in last dim

# Pipeline: beamform + envelope_detect only (no normalize / log_compress)
pipeline = Pipeline(
    operations=[
        Beamform(
            beamformer="delay_and_sum",
            enable_pfield=True,
            num_patches=100,
        ),
        EnvelopeDetect(),
    ],
    with_batch_dim=False,
    jit_options="pipeline",
)
parameters = pipeline.prepare_parameters(probe, scan)
parameters.pop("dynamic_range", None)

print(f"Loaded 1 frame. Raw shape: {data.shape}")
print(f"Scan xlims: {[round(v * 1e3, 1) for v in scan.xlims]} mm")
print(f"Scan zlims: {[round(v * 1e3, 1) for v in scan.zlims]} mm")

zea: Loading cached result for compute_pfield.
zea: WARNING No transmit origins provided, using zeros
zea: WARNING No transmit waveform indices provided, using zeros


Loaded 1 frame. Raw shape: (1, 149, 2176, 128, 1)
Scan xlims: [np.float32(-19.1), np.float32(19.1)] mm
Scan zlims: [0.0, 40.0] mm


In [5]:
# Beamform + envelope-detect the single selected frame
out = pipeline(data=data[0, scan.selected_transmits], **parameters)
envelope = np.array(out[pipeline.output_key])  # (H, W)

# Keep a batch dimension for model inference: (N, H, W) with N=1
envelopes = envelope[np.newaxis, ...]
print(f"Envelope shape: {envelopes.shape}  range: [{envelopes.min():.3g}, {envelopes.max():.3g}]")

zea: DEBUG [zea.Pipeline] The following input keys are not used by the pipeline: {'n_el', 'zlims', 'center_frequency', 'xlims'}. Make sure this is intended. This warning will only be shown once.


Envelope shape: (1, 812, 774)  range: [0.0789, 4.4e+04]


## Load Speckle2Self model

`Speckle2Self` can be loaded directly from Hugging Face:

In [6]:
model = Speckle2Self.from_preset("hf://zeahub/speckle2self-invivo")
print("Model loaded:", model)

Model loaded: <Speckle2Self name=speckle2_self, built=True>


## Preprocess and run inference

The model expects **linear-scale envelope data** as input (not log-compressed)

In [7]:
def linear_normalize(img: np.ndarray) -> np.ndarray:
    """Per-image min–max normalisation to [0, 1]."""
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn + 1e-16)


def to_bmode(img: np.ndarray, dynamic_range: tuple = (-60, 0)) -> np.ndarray:
    """Log-compress a linear envelope to a display B-mode image.

    Normalises to [0, 1] first, then applies 20 log10, clips to
    ``dynamic_range`` dB, and rescales to [0, 1].
    """
    img_n = linear_normalize(img)
    db = 20 * np.log10(np.clip(img_n, 1e-6, None))
    db = np.clip(db, dynamic_range[0], dynamic_range[1])
    return (db - dynamic_range[0]) / (dynamic_range[1] - dynamic_range[0])


# Per-image normalisation → [N, 1, H, W] model input (matches original inference.py)
envelopes_norm = np.stack([linear_normalize(e) for e in envelopes])  # (N, H, W)
model_input = envelopes_norm[:, np.newaxis, :, :].astype(np.float32)  # (N, 1, H, W)

# Run speckle reduction
despeckled = model(model_input)  # (N, 1, H, W), values in [0, 1]
despeckled = np.array(despeckled)[:, 0]  # (N, H, W)

print(f"Input  {model_input.shape}  mean={envelopes_norm.mean():.3f}")
print(f"Output {despeckled.shape}  mean={despeckled.mean():.3f}  max={despeckled.max():.3f}")

Input  (1, 1, 812, 774)  mean=0.035
Output (1, 812, 774)  mean=0.145  max=0.905


## Visualize results

For a simple comparison, we display one frame before and after Speckle2Self despeckling.


In [8]:
xlims_mm = [v * 1e3 for v in scan.xlims]
zlims_mm = [v * 1e3 for v in scan.zlims]
extent = [xlims_mm[0], xlims_mm[1], zlims_mm[1], zlims_mm[0]]

before = to_bmode(envelopes[0], dynamic_range)
after = despeckled[0]

fig, axes = plt.subplots(1, 2, figsize=(8, 4), squeeze=False)

axes[0, 0].imshow(before, cmap="gray", vmin=0, vmax=1, extent=extent)
axes[0, 0].set_title("B-Mode")
axes[0, 0].set_xlabel("X (mm)")
axes[0, 0].set_ylabel("Z (mm)")

axes[0, 1].imshow(after, cmap="gray", vmin=0, vmax=1, extent=extent)
axes[0, 1].set_title("Despeckled B-Mode (Speckle2Self)")
axes[0, 1].set_xlabel("X (mm)")
axes[0, 1].set_ylabel("Z (mm)")

plt.tight_layout()
plt.savefig("speckle2self_output.png", bbox_inches="tight", dpi=100)
plt.close()

**Speckle2Self result on in-vivo carotid data**

![Image](./speckle2self_output.png)